# PINN-DPV — run everything from here (Colab)

The one notebook to open on Colab. It:

1. mounts Google Drive — **every output lands there** (`MyDrive/pinn-dpv/`);
2. clones only what runs from the public repository `smindiol/pinn-dpv` (newest commit, library and step notebooks);
3. installs JAX with CUDA and the pinned dependencies;
4. downloads the dPV dataset from Kaggle;
5. runs the declared steps in order — `sigma`, `pilot`, `report` — each as its own
   notebook, printing progress and the estimated time left as it goes, and saves
   each executed notebook to Drive.

Interrupted? Run it again: finished sigma files are skipped and every pilot run
resumes from its last checkpoint on Drive.

**Before running**

- Runtime → Change runtime type → **A100 GPU**.
- Colab Secrets (key icon on the left), with notebook access enabled:
  - `KAGGLE_USERNAME` and `KAGGLE_KEY` (or `KAGGLE_API_TOKEN`) of an account that can read the dataset;
  - optionally `GH_TOKEN`, a fine-grained token with contents write on `smindiol/pinn-dpv` only, to also push light results to the `results` branch.

Never paste a token into a cell.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/smindiol/pinn-dpv.git"
BRANCH = "main"
KAGGLE_DATASET = "sgarizado/dpv-pinn-canonical"   # owner/slug of the dPV files on Kaggle
DRIVE_ROOT = "/content/drive/MyDrive/pinn-dpv"             # every output goes here
MODE = "rehearsal"            # "rehearsal": small, minutes; "full": E0 at the protocol's scale
STEPS = ["sigma", "pilot", "report"]
SIGMA_FAMILY_STAGES = "E1A,E3A"   # what E0 needs; "" computes all eight family-stages

CLONE = "/content/pinn-dpv"
DATA = "/content/dpv-data"


## 2. Drive and GPU

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
      or "no GPU: Runtime -> Change runtime type")

## 3. Repository

In [ ]:
# Only what runs: the newest commit (--depth 1), no history, and only the library,
# the step notebooks and the root files (requirements) -- not tests, docs or results.
SPARSE = ["src", "PINN-DPV/Notebooks"]
if os.path.isdir(os.path.join(CLONE, ".git")):
    subprocess.run(["git", "-C", CLONE, "fetch", "-q", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", CLONE, "reset", "-q", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", "-b", BRANCH,
                    REPO_URL, CLONE], check=True)
subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", *SPARSE], check=True)
COMMIT = subprocess.check_output(["git", "-C", CLONE, "rev-parse", "HEAD"], text=True).strip()
print("commit", COMMIT)

## 4. Dependencies

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax[cuda12]==0.4.38", "optax==0.2.4",
                "chex==0.1.88", "h5py>=3.10", "optuna>=3.6", "scipy>=1.15", "papermill", "kagglehub"], check=True)
# Checked in a child process: importing JAX here would preallocate most of the GPU
# memory in this kernel, and each step notebook runs in a kernel of its own.
probe = subprocess.run([sys.executable, "-c", "import jax; d = jax.devices(); print(d); assert d[0].platform == 'gpu'"],
                       capture_output=True, text=True)
print(probe.stdout or probe.stderr[-2000:])
assert probe.returncode == 0, "JAX does not see the GPU: Runtime -> Change runtime type -> A100"


## 5. Dataset from Kaggle

In [ ]:
import glob, shutil
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

for name in ("KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN"):
    if secret(name):
        os.environ[name] = secret(name)

def data_root(base):
    hit = sorted(glob.glob(os.path.join(base, "**", "dpv_E1A_b000.h5"), recursive=True))
    if not hit:
        return None
    here = os.path.dirname(hit[0])   # flat on Kaggle; <run>/<file> in a dPV checkout
    return os.path.dirname(here) if os.path.basename(here) == "s1" else here

root = data_root(DATA)
if root is None:
    import kagglehub
    path = kagglehub.dataset_download(KAGGLE_DATASET)
    root = data_root(path)
    assert root, f"no dpv_E1A_b000.h5 inside the download at {path}"
os.environ["PINN_DPV_DATA"] = root
print("dataset:", root)
print(subprocess.run(["du", "-sh", root], capture_output=True, text=True).stdout)

## 6. Environment the step notebooks read

In [ ]:
os.environ.update({
    "FORGE_CLONE_ROOT": CLONE,          # the step notebooks' first cell proves this checkout is at COMMIT
    "FORGE_CLONE_COMMIT": COMMIT,
    "PINN_DPV_RESULTS": os.path.join(DRIVE_ROOT, "Results"),
    "PINN_DPV_REHEARSAL": "0" if MODE == "full" else "1",
    "PINN_DPV_FAMILY_STAGES": SIGMA_FAMILY_STAGES,
})
token = secret("GH_TOKEN")
if token:
    os.environ["PINN_DPV_GH_TOKEN"] = token
print({k: v for k, v in os.environ.items() if k.startswith(("PINN_DPV", "FORGE_")) and "TOKEN" not in k})
print("push to GitHub:", bool(token))

## 7. Run the steps

Each step notebook runs in its own kernel; its prints stream here (progress, voltage,
time elapsed and remaining) and the executed copy is saved to
`MyDrive/pinn-dpv/executed/`.

In [ ]:
import time
import papermill as pm

NOTEBOOKS = os.path.join(CLONE, "PINN-DPV", "Notebooks")
EXECUTED = os.path.join(DRIVE_ROOT, "executed", time.strftime("%Y%m%d-%H%M%S") + f"-{MODE}")
os.makedirs(EXECUTED, exist_ok=True)
t0 = time.time()
for i, step in enumerate(STEPS, 1):
    print(f"\n=== step {i}/{len(STEPS)}: {step}  (total elapsed {(time.time() - t0) / 60:.1f} min) ===", flush=True)
    pm.execute_notebook(os.path.join(NOTEBOOKS, f"{step}.ipynb"), os.path.join(EXECUTED, f"{step}.ipynb"),
                        cwd=NOTEBOOKS, kernel_name="python3", log_output=True, progress_bar=False)
print(f"\nall steps done in {(time.time() - t0) / 60:.1f} min; executed notebooks in {EXECUTED}")

## 8. E0 summary

In [ ]:
import json

rep = json.load(open(os.path.join(DRIVE_ROOT, "Results", "report", "e0.json")))
for fs, s in rep["sigma"].items():
    print(f"{fs}: r = {s['r_pp']:.4f} pp")
for r in rep["pilot"]:
    print(f"{r['run_id']:<50} val {r.get('mean_e_pce_val', float('nan')):.4f} pp  test {r.get('mean_e_pce_test', float('nan')):.4f} pp  {r.get('seconds', 0) / 60:.1f} min")
for fs, b in rep["budget"].items():
    print(f"budget {fs}: {b['iterations']} iterations (seeds {b['seeds']})")